# Stock Direction Predictor (v2)
Predicts whether a stock will be **UP or DOWN in 3 days** using XGBoost, lag features, and market context (S&P 500 + VIX).

Accuracy is measured with **walk-forward cross-validation** — no lookahead bias.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

TICKER   = 'AAPL'   # change to any ticker
PERIOD   = '5y'
HORIZON  = 3        # predict direction N days out
N_SPLITS = 5

plt.style.use('dark_background')

In [ ]:
raw = yf.download([TICKER, '^GSPC', '^VIX'], period=PERIOD, auto_adjust=True, progress=False)
raw.columns = ['_'.join(c).strip() for c in raw.columns]
print(f'Downloaded {len(raw)} rows')
raw.tail(3)

In [ ]:
c   = raw[f'Close_{TICKER}']
vol = raw[f'Volume_{TICKER}']
sp  = raw['Close_^GSPC']
vix = raw['Close_^VIX']

df = pd.DataFrame(index=raw.index)

# technical indicators
df['sma_5']   = c.rolling(5).mean()
df['sma_20']  = c.rolling(20).mean()
df['sma_50']  = c.rolling(50).mean()
df['ema_12']  = c.ewm(span=12).mean()
df['ema_26']  = c.ewm(span=26).mean()
df['macd']    = df['ema_12'] - df['ema_26']

delta = c.diff()
gain  = delta.clip(lower=0).rolling(14).mean()
loss  = (-delta.clip(upper=0)).rolling(14).mean()
df['rsi'] = 100 - (100 / (1 + gain / loss))

rolling_std       = c.rolling(20).std()
bb_upper          = df['sma_20'] + 2 * rolling_std
bb_lower          = df['sma_20'] - 2 * rolling_std
df['bb_position'] = (c - bb_lower) / (bb_upper - bb_lower)
df['bb_width']    = (bb_upper - bb_lower) / df['sma_20']
df['daily_range'] = (raw[f'High_{TICKER}'] - raw[f'Low_{TICKER}']) / c
df['momentum_10'] = c - c.shift(10)
df['vol_ratio']   = vol / vol.rolling(20).mean()

# lag features
for d in [1, 2, 3, 5, 10]:
    df[f'ret_{d}d'] = c.pct_change(d)
df['vol_chg_1d'] = vol.pct_change(1)
df['vol_chg_5d'] = vol.pct_change(5)
df['gap'] = (raw[f'Open_{TICKER}'] - c.shift(1)) / c.shift(1)

# market context
df['sp500_ret_1d']    = sp.pct_change(1)
df['sp500_ret_5d']    = sp.pct_change(5)
df['sp500_vs_sma20']  = sp / sp.rolling(20).mean() - 1
df['vix']             = vix
df['vix_chg_1d']      = vix.pct_change(1)

# target
df['target'] = (c.shift(-HORIZON) > c).astype(int)
df = df.dropna()

print(f'{len(df)} rows after feature engineering')
df.tail(3)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(df.index, c[df.index], color='white', linewidth=1, label='Close')
ax1.plot(df.index, df['sma_20'], color='#f59e0b', linewidth=0.8, alpha=0.8, label='SMA 20')
ax1.plot(df.index, df['sma_50'], color='#6366f1', linewidth=0.8, alpha=0.8, label='SMA 50')
ax1.fill_between(df.index, bb_upper[df.index], bb_lower[df.index], alpha=0.08, color='white', label='Bollinger Bands')
ax1.set_title(f'{TICKER} — Price & Indicators', fontsize=13, pad=10)
ax1.set_ylabel('Price (USD)')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.1)

ax2.plot(df.index, df['rsi'], color='#ec4899', linewidth=0.8)
ax2.axhline(70, color='white', linestyle='--', linewidth=0.5, alpha=0.3)
ax2.axhline(30, color='white', linestyle='--', linewidth=0.5, alpha=0.3)
ax2.set_ylabel('RSI')
ax2.set_ylim(0, 100)
ax2.grid(alpha=0.1)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

In [ ]:
features = [col for col in df.columns if col != 'target']
X = df[features].values
y = df['target'].values

model = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    verbosity=0,
)

tscv = TimeSeriesSplit(n_splits=N_SPLITS)
fold_scores, all_preds, all_true = [], [], []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    model.fit(X[train_idx], y[train_idx])
    preds = model.predict(X[test_idx])
    score = accuracy_score(y[test_idx], preds)
    fold_scores.append(score)
    all_preds.extend(preds)
    all_true.extend(y[test_idx])
    print(f'  Fold {fold+1}: {score:.1%}')

overall_acc = accuracy_score(all_true, all_preds)
print(f'\nOverall walk-forward accuracy: {overall_acc:.1%}')

# refit on all data for final prediction
model.fit(X, y)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

cm = confusion_matrix(all_true, all_preds)
ConfusionMatrixDisplay(cm, display_labels=['DOWN', 'UP']).plot(ax=ax1, colorbar=False, cmap='Blues')
ax1.set_title('Confusion Matrix (walk-forward)')

importances = pd.Series(model.feature_importances_, index=features).sort_values().tail(15)
importances.plot(kind='barh', ax=ax2, color='#6366f1')
ax2.set_title('Top 15 Feature Importances')
ax2.grid(alpha=0.15, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
latest = df[features].iloc[[-1]]
prob   = model.predict_proba(latest.values)[0]
direction = 'UP ↑' if prob[1] >= 0.5 else 'DOWN ↓'

print(f'\nTicker              : {TICKER}')
print(f'Last close          : ${c.iloc[-1]:.2f}')
print(f'Prediction ({HORIZON}d out) : {direction}')
print(f'Confidence          : {max(prob):.1%}')